In [1]:
# ======================================================
# RESETAR AMBIENTE COM VERSÕES COMPATÍVEIS
# ======================================================

!pip uninstall -y unbabel-comet transformers pytorch-lightning torchmetrics

# Instalar versões compatíveis
!pip install -q \
    "pip<24.1" \
    "transformers==4.40.2" \
    "unbabel-comet>=2.2.3" \
    sentencepiece

Found existing installation: transformers 5.0.0
Uninstalling transformers-5.0.0:
  Successfully uninstalled transformers-5.0.0
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 138.0/138.0 kB 3.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 55.5/55.5 kB 3.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 3.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.0/9.0 MB 61.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.1/2.1 MB 63.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 91.0/91.0 kB 6.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 101.4/101.4 kB 6.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 32.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.0/18.0 MB 78.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 295.2/295.2 kB 18.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 852.4/852.4 kB 35.8 MB/s eta 0:00:0

Reinicie o Ambiente depois da Célula acima

In [1]:
# ======================================================
# IMPORTS
# ======================================================

import json
import pandas as pd
import torch

from google.colab import files
from comet import download_model, load_from_checkpoint

In [2]:
# ======================================================
# UPLOAD DO JSON EM INGLÊS
# ======================================================

print("Envie o arquivo JSON ORIGINAL em inglês")

uploaded_en = files.upload()

if len(uploaded_en) != 1:
    raise Exception(
        "Envie apenas UM arquivo para o dataset em inglês."
    )

english_file = list(uploaded_en.keys())[0]

print(f"\nArquivo em inglês carregado: {english_file}")

# ======================================================
# UPLOAD DO JSON EM PORTUGUÊS
# ======================================================

print("\nEnvie o arquivo JSON TRADUZIDO para português")

uploaded_pt = files.upload()

if len(uploaded_pt) != 1:
    raise Exception(
        "Envie apenas UM arquivo para o dataset em português."
    )

portuguese_file = list(uploaded_pt.keys())[0]

print(f"\nArquivo em português carregado: {portuguese_file}")

# ======================================================
# CARREGAR JSONS
# ======================================================

with open(english_file, "r", encoding="utf-8") as f:
    english_data = json.load(f)

with open(portuguese_file, "r", encoding="utf-8") as f:
    portuguese_data = json.load(f)

# ======================================================
# VALIDAÇÃO
# ======================================================

if len(english_data) != len(portuguese_data):
    raise Exception(
        "Os arquivos possuem quantidades diferentes de registros."
    )

print(f"\nQuantidade de registros: {len(english_data)}")

# ======================================================
# PREPARAR DADOS PARA O COMETKiwi
# ======================================================

data = []

results = []

for idx, (en, pt) in enumerate(zip(english_data, portuguese_data)):

    # Campos esperados:
    # instruction
    # input
    # output

    en_instruction = str(en.get("instruction", ""))
    en_input = str(en.get("input", ""))
    en_output = str(en.get("output", ""))

    pt_instruction = str(pt.get("instruction", ""))
    pt_input = str(pt.get("input", ""))
    pt_output = str(pt.get("output", ""))

    # Texto fonte em inglês
    source_text = (
        f"Instruction: {en_instruction}\n"
        f"Input: {en_input}\n"
        f"Output: {en_output}"
    )

    # Tradução em português
    translated_text = (
        f"Instrução: {pt_instruction}\n"
        f"Entrada: {pt_input}\n"
        f"Saída: {pt_output}"
    )

    data.append({
        "src": source_text,
        "mt": translated_text
    })

    results.append({
        "id": idx,
        "source": source_text,
        "translation": translated_text
    })

# ======================================================
# LOGIN NO HUGGING FACE
# ======================================================

from huggingface_hub import login

# Cole aqui seu token do Hugging Face
from dotenv import load_dotenv
import os
load_dotenv()
HF_TOKEN = os.getenv("HF_TOKEN")

login(token=HF_TOKEN)

# ======================================================
# BAIXAR MODELO COMETKiwi
# ======================================================

print("\nBaixando modelo COMETKiwi...")

model_path = download_model(
    "Unbabel/wmt22-cometkiwi-da"
)

# ======================================================
# CARREGAR MODELO
# ======================================================

print("Carregando modelo...")

model = load_from_checkpoint(model_path)

# ======================================================
# VERIFICAR GPU
# ======================================================

use_gpu = torch.cuda.is_available()

print(f"\nGPU disponível: {use_gpu}")

# ======================================================
# EXECUTAR AVALIAÇÃO
# ======================================================

print("\nExecutando avaliação...")

model_output = model.predict(
    data,
    batch_size=4,
    accelerator="gpu" if use_gpu else "cpu"
)

scores = model_output.scores

# ======================================================
# ADICIONAR SCORES
# ======================================================

for i in range(len(results)):
    results[i]["cometkiwi_score"] = scores[i]

# ======================================================
# CLASSIFICAÇÃO OPCIONAL
# ======================================================

def classify(score):
    if score >= 0.85:
        return "excelente"
    elif score >= 0.75:
        return "boa"
    elif score >= 0.60:
        return "mediana"
    else:
        return "ruim"

for item in results:
    item["quality"] = classify(item["cometkiwi_score"])

# ======================================================
# DATAFRAME FINAL
# ======================================================

df = pd.DataFrame(results)

# ======================================================
# ESTATÍSTICAS
# ======================================================

mean_score = df["cometkiwi_score"].mean()
min_score = df["cometkiwi_score"].min()
max_score = df["cometkiwi_score"].max()

print("\n===== RESULTADOS =====")
print(f"Média COMETKiwi : {mean_score:.4f}")
print(f"Menor score     : {min_score:.4f}")
print(f"Maior score     : {max_score:.4f}")

# ======================================================
# EXIBIR TRADUÇÕES PROBLEMÁTICAS
# ======================================================

print("\n===== POSSÍVEIS PROBLEMAS =====")

low_quality = df[df["cometkiwi_score"] < 0.60]

if len(low_quality) == 0:
    print("Nenhuma tradução problemática encontrada.")
else:
    for _, row in low_quality.head(5).iterrows():
        print("\n-----------------------------------")
        print(f"ID: {row['id']}")
        print(f"Score: {row['cometkiwi_score']:.4f}")
        print(f"Qualidade: {row['quality']}")
        print("\nFonte:")
        print(row["source"][:500])
        print("\nTradução:")
        print(row["translation"][:500])

# ======================================================
# SALVAR RESULTADOS
# ======================================================

output_csv = "cometkiwi_translation_evaluation.csv"

df.to_csv(output_csv, index=False)

print(f"\nArquivo salvo: {output_csv}")

# ======================================================
# DOWNLOAD AUTOMÁTICO
# ======================================================

files.download(output_csv)

Envie o arquivo JSON ORIGINAL em inglês


Saving crafted_instruction_data_qa.json to crafted_instruction_data_qa.json

Arquivo em inglês carregado: crafted_instruction_data_qa.json

Envie o arquivo JSON TRADUZIDO para português


Saving nllb_dataset_traduzido_pt.json to nllb_dataset_traduzido_pt.json

Arquivo em português carregado: nllb_dataset_traduzido_pt.json

Quantidade de registros: 2000

Baixando modelo COMETKiwi...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Fetching 5 files:   0%|          | 0/5 [00:00<?, ?it/s]

...


GPU disponível: False

Executando avaliação...


Predicting DataLoader 0: 100%|██████████| 500/500 [4:23:33<00:00, 31.63s/it]



===== RESULTADOS =====
Média COMETKiwi : 0.6296
Menor score     : 0.2983
Maior score     : 0.8603

===== POSSÍVEIS PROBLEMAS =====

-----------------------------------
ID: 0
Score: 0.5761
Qualidade: ruim

Fonte:
Instruction: Who was the first Spanish actor to win an Oscar, for his role in the film No Country for Old Men?
Input: [DOC] [TLE] Javier Bardem becomes first Spanish actor to win Oscar ...Javier Bardem becomes first Spanish actor to win Oscar | Reuters [PAR] Mon Feb 25, 2008 | 9:32 AM EST [PAR] Javier Bardem becomes first Spanish actor to win Oscar [PAR] 1/3 [PAR] Best supporting actor nominee Javier Bardem of the film ''No Country for Old Men'' arrives at the 80th annual Academy Awards, the Osc

Tradução:
Instrução: Quem foi o primeiro ator espanhol a ganhar um Oscar, pelo seu papel no filme Não há país para velhos homens?
Entrada: [TLE] Javier Bardem se torna o primeiro ator espanhol a ganhar um Oscar ...Javier Bardem se torna o primeiro ator espanhol a ganhar um Oscar. [PAR

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [3]:
# ======================================================
# CONTAGEM POR FAIXAS DE SCORE
# ======================================================

accepted_count = len(
    df[df["cometkiwi_score"] > 0.80]
)

optional_review_count = len(
    df[
        (df["cometkiwi_score"] >= 0.60) &
        (df["cometkiwi_score"] <= 0.80)
    ]
)

mandatory_review_count = len(
    df[df["cometkiwi_score"] < 0.60]
)

total_count = len(df)

# ======================================================
# EXIBIR RESULTADOS
# ======================================================

print("\n===== DISTRIBUIÇÃO POR FAIXAS =====\n")

print(f"> 0.80")
print(f"Alto: {accepted_count}")
print(
    f"Percentual: "
    f"{(accepted_count / total_count) * 100:.2f}%"
)

print("\n0.60 – 0.80")
print(f"Médio: {optional_review_count}")
print(
    f"Percentual: "
    f"{(optional_review_count / total_count) * 100:.2f}%"
)

print("\n< 0.60")
print(f"Baixo: {mandatory_review_count}")
print(
    f"Percentual: "
    f"{(mandatory_review_count / total_count) * 100:.2f}%"
)


===== DISTRIBUIÇÃO POR FAIXAS =====

> 0.80
Alto: 215
Percentual: 10.75%

0.60 – 0.80
Médio: 885
Percentual: 44.25%

< 0.60
Baixo: 900
Percentual: 45.00%


In [1]:
# ======================================================
# UPLOAD DO CSV DE RESULTADOS COMETKiwi
# ======================================================

from google.colab import files
import pandas as pd
import json

print("Envie o arquivo CSV gerado pelo COMETKiwi")

uploaded_csv = files.upload()

if len(uploaded_csv) != 1:
    raise Exception("Envie apenas um arquivo CSV.")

csv_file = list(uploaded_csv.keys())[0]

print(f"\nCSV carregado: {csv_file}")

# ======================================================
# CARREGAR CSV
# ======================================================

df_scores = pd.read_csv(csv_file)

print("\nQuantidade original de registros:")
print(len(df_scores))

# ======================================================
# FILTRAR SCORES >= 0.60
# ======================================================

filtered_df = df_scores[
    df_scores["cometkiwi_score"] >= 0.60
].copy()

print("\nQuantidade após filtragem:")
print(len(filtered_df))

removed_count = len(df_scores) - len(filtered_df)

print(f"\nRegistros removidos: {removed_count}")

# ======================================================
# RECALCULAR MÉTRICAS
# ======================================================

new_mean = filtered_df["cometkiwi_score"].mean()
new_min = filtered_df["cometkiwi_score"].min()
new_max = filtered_df["cometkiwi_score"].max()

print("\n===== NOVOS RESULTADOS =====")
print(f"Nova média COMETKiwi : {new_mean:.4f}")
print(f"Novo menor score     : {new_min:.4f}")
print(f"Novo maior score     : {new_max:.4f}")

# ======================================================
# DISTRIBUIÇÃO POR FAIXAS
# ======================================================

accepted_count = len(
    filtered_df[filtered_df["cometkiwi_score"] > 0.80]
)

optional_review_count = len(
    filtered_df[
        (filtered_df["cometkiwi_score"] >= 0.60) &
        (filtered_df["cometkiwi_score"] <= 0.80)
    ]
)

total_count = len(filtered_df)

print("\n===== DISTRIBUIÇÃO ATUALIZADA =====\n")

print(f"> 0.80")
print(f"Aceitar automaticamente : {accepted_count}")
print(
    f"Percentual               : "
    f"{(accepted_count / total_count) * 100:.2f}%"
)

print("\n0.60 – 0.80")
print(f"Revisão opcional         : {optional_review_count}")
print(
    f"Percentual               : "
    f"{(optional_review_count / total_count) * 100:.2f}%"
)

# ======================================================
# SALVAR CSV FILTRADO
# ======================================================

filtered_csv = "filtered_cometkiwi_translation_evaluation.csv"

filtered_df.to_csv(filtered_csv, index=False)

print(f"\nArquivo filtrado salvo: {filtered_csv}")

# ======================================================
# DOWNLOAD AUTOMÁTICO
# ======================================================

files.download(filtered_csv)

Envie o arquivo CSV gerado pelo COMETKiwi


Saving nllb_cometkiwi_translation_evaluation.csv to nllb_cometkiwi_translation_evaluation.csv

CSV carregado: nllb_cometkiwi_translation_evaluation.csv

Quantidade original de registros:
2000

Quantidade após filtragem:
1100

Registros removidos: 900

===== NOVOS RESULTADOS =====
Nova média COMETKiwi : 0.7224
Novo menor score     : 0.6002
Novo maior score     : 0.8603

===== DISTRIBUIÇÃO ATUALIZADA =====

> 0.80
Aceitar automaticamente : 215
Percentual               : 19.55%

0.60 – 0.80
Revisão opcional         : 885
Percentual               : 80.45%

Arquivo filtrado salvo: filtered_cometkiwi_translation_evaluation.csv


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [2]:
# ======================================================
# UPLOAD DO CSV DE RESULTADOS COMETKiwi
# ======================================================

from google.colab import files
import pandas as pd
import json

print("Envie o arquivo CSV gerado pelo COMETKiwi")

uploaded_csv = files.upload()

if len(uploaded_csv) != 1:
    raise Exception("Envie apenas um arquivo CSV.")

csv_file = list(uploaded_csv.keys())[0]

print(f"\nCSV carregado: {csv_file}")

# ======================================================
# CARREGAR CSV
# ======================================================

df_scores = pd.read_csv(csv_file)

print("\nQuantidade original de registros:")
print(len(df_scores))

# ======================================================
# FILTRAR SCORES < 0.60
# (mantém apenas os problemáticos)
# ======================================================

filtered_df = df_scores[
    df_scores["cometkiwi_score"] < 0.60
].copy()

print("\nQuantidade após filtragem:")
print(len(filtered_df))

removed_count = len(df_scores) - len(filtered_df)

print(f"\nRegistros removidos (>= 0.60): {removed_count}")

# ======================================================
# RECALCULAR MÉTRICAS
# ======================================================

new_mean = filtered_df["cometkiwi_score"].mean()
new_min = filtered_df["cometkiwi_score"].min()
new_max = filtered_df["cometkiwi_score"].max()

print("\n===== NOVOS RESULTADOS =====")
print(f"Nova média COMETKiwi : {new_mean:.4f}")
print(f"Novo menor score     : {new_min:.4f}")
print(f"Novo maior score     : {new_max:.4f}")

# ======================================================
# DISTRIBUIÇÃO POR FAIXAS
# ======================================================

mandatory_review_count = len(
    filtered_df[filtered_df["cometkiwi_score"] < 0.60]
)

total_count = len(filtered_df)

print("\n===== DISTRIBUIÇÃO ATUALIZADA =====\n")

print("< 0.60")
print(f"Revisão obrigatória      : {mandatory_review_count}")

print(
    f"Percentual               : "
    f"{(mandatory_review_count / total_count) * 100:.2f}%"
)

# ======================================================
# SALVAR CSV FILTRADO
# ======================================================

filtered_csv = "low_score_cometkiwi_translation_evaluation.csv"

filtered_df.to_csv(filtered_csv, index=False)

print(f"\nArquivo filtrado salvo: {filtered_csv}")

# ======================================================
# DOWNLOAD AUTOMÁTICO
# ======================================================

files.download(filtered_csv)

Envie o arquivo CSV gerado pelo COMETKiwi


Saving nllb_cometkiwi_translation_evaluation.csv to nllb_cometkiwi_translation_evaluation (1).csv

CSV carregado: nllb_cometkiwi_translation_evaluation (1).csv

Quantidade original de registros:
2000

Quantidade após filtragem:
900

Registros removidos (>= 0.60): 1100

===== NOVOS RESULTADOS =====
Nova média COMETKiwi : 0.5161
Novo menor score     : 0.2983
Novo maior score     : 0.5998

===== DISTRIBUIÇÃO ATUALIZADA =====

< 0.60
Revisão obrigatória      : 900
Percentual               : 100.00%

Arquivo filtrado salvo: low_score_cometkiwi_translation_evaluation.csv


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [4]:
# ======================================================
# GERAR JSONS FILTRADOS
# ======================================================

from google.colab import files
import json
import pandas as pd

# ======================================================
# UPLOAD DOS JSONS ORIGINAIS
# ======================================================

print("Envie novamente o JSON ORIGINAL em inglês")

uploaded_en = files.upload()

english_file = list(uploaded_en.keys())[0]

print(f"\nArquivo carregado: {english_file}")

print("\nEnvie novamente o JSON TRADUZIDO em português")

uploaded_pt = files.upload()

portuguese_file = list(uploaded_pt.keys())[0]

print(f"\nArquivo carregado: {portuguese_file}")

# ======================================================
# UPLOAD DO CSV FILTRADO
# ======================================================

print("\nEnvie o CSV filtrado gerado anteriormente")

uploaded_csv = files.upload()

csv_file = list(uploaded_csv.keys())[0]

print(f"\nCSV carregado: {csv_file}")

# ======================================================
# CARREGAR ARQUIVOS
# ======================================================

with open(english_file, "r", encoding="utf-8") as f:
    english_data = json.load(f)

with open(portuguese_file, "r", encoding="utf-8") as f:
    portuguese_data = json.load(f)

filtered_df = pd.read_csv(csv_file)

# ======================================================
# OBTER IDS MANTIDOS
# ======================================================

valid_ids = set(filtered_df["id"].tolist())

print(f"\nQuantidade de registros mantidos: {len(valid_ids)}")

# ======================================================
# FILTRAR JSONS
# ======================================================

filtered_english = []
filtered_portuguese = []

for idx, (en_item, pt_item) in enumerate(
    zip(english_data, portuguese_data)
):
    if idx in valid_ids:
        filtered_english.append(en_item)
        filtered_portuguese.append(pt_item)

# ======================================================
# EXIBIR ESTATÍSTICAS
# ======================================================

print("\n===== RESULTADOS =====")
print(f"JSON inglês original     : {len(english_data)}")
print(f"JSON português original  : {len(portuguese_data)}")

print(f"\nJSON inglês filtrado     : {len(filtered_english)}")
print(f"JSON português filtrado  : {len(filtered_portuguese)}")

removed = len(english_data) - len(filtered_english)

print(f"\nRegistros removidos      : {removed}")

# ======================================================
# SALVAR JSONS FILTRADOS
# ======================================================

filtered_english_file = "low_filtered_english_dataset.json"
filtered_portuguese_file = "low_filtered_portuguese_dataset.json"

with open(filtered_english_file, "w", encoding="utf-8") as f:
    json.dump(
        filtered_english,
        f,
        ensure_ascii=False,
        indent=2
    )

with open(filtered_portuguese_file, "w", encoding="utf-8") as f:
    json.dump(
        filtered_portuguese,
        f,
        ensure_ascii=False,
        indent=2
    )

print("\nArquivos JSON filtrados salvos.")

# ======================================================
# DOWNLOAD AUTOMÁTICO
# ======================================================

files.download(filtered_english_file)
files.download(filtered_portuguese_file)

Envie novamente o JSON ORIGINAL em inglês


Saving crafted_instruction_data_qa.json to crafted_instruction_data_qa (1).json

Arquivo carregado: crafted_instruction_data_qa (1).json

Envie novamente o JSON TRADUZIDO em português


Saving nllb_dataset_traduzido_pt.json to nllb_dataset_traduzido_pt (1).json

Arquivo carregado: nllb_dataset_traduzido_pt (1).json

Envie o CSV filtrado gerado anteriormente


Saving nllb_low_score_cometkiwi_translation_evaluation.csv to nllb_low_score_cometkiwi_translation_evaluation.csv

CSV carregado: nllb_low_score_cometkiwi_translation_evaluation.csv

Quantidade de registros mantidos: 900

===== RESULTADOS =====
JSON inglês original     : 2000
JSON português original  : 2000

JSON inglês filtrado     : 900
JSON português filtrado  : 900

Registros removidos      : 1100

Arquivos JSON filtrados salvos.


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>